In [ ]:
# week 4
# imports
import os
import io
import sys
import json
import shutil
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Load environment variables
load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("No OpenRouter API key found")
else:
    print("OpenRouter API key loaded")

In [ ]:
# Models
MODELS = {
    "OpenRouter: gpt-4o-mini": ("openai/gpt-4o-mini", "https://openrouter.ai/api/v1", api_key),
    "OpenRouter: claude-3-haiku": ("anthropic/claude-3-haiku", "https://openrouter.ai/api/v1", api_key),
}

In [ ]:
#Client Setup
def get_client(model_key):
    model_id, base_url, key = MODELS.get(model_key)
    return OpenAI(api_key=key, base_url=base_url), model_id

In [ ]:
# System Prompt
SYSTEM_PROMPT = """
You are an expert software engineer.

Convert Python code into high-performance JavaScript.

Rules:
- Output ONLY JavaScript code
- The JavaScript must produce the same output
- Use modern JavaScript (ES6+)
- Use console.log instead of print
"""

In [ ]:
# Create User Prompt
def create_user_prompt(python_code):
    
    return f"""
Convert the following Python code to JavaScript.

Requirements:
- Output must be identical
- Use console.log for output
- JavaScript must run in Node.js

Python Code:

{python_code}
"""

In [ ]:
# Conversion Function
def convert_to_javascript(model_key, python_code):

    if not python_code.strip():
        return "// No code provided"

    client, model_id = get_client(model_key)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": create_user_prompt(python_code)}
    ]

    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=messages
        )

        js_code = response.choices[0].message.content.strip()

        return js_code

    except Exception as e:
        return f"// Error: {e}"

In [ ]:
# Run Python Code
def run_python(code):

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"

    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
# Run JavaScript Code
def run_javascript(code):

    if not shutil.which("node"):
        return "Node.js not installed"

    try:
        with open("main.js", "w") as f:
            f.write(code)

        result = subprocess.run(
            ["node", "main.js"],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:
            return result.stderr

        return result.stdout

    except Exception as e:
        return str(e)

In [ ]:
# Example Python Code
EXAMPLE_CODE = """
def fibonacci(n):
    a, b = 0, 1

    for i in range(n):
        a, b = b, a + b

    print(a)

fibonacci(10)
"""

In [ ]:
# Gradio Interface
with gr.Blocks(title="Week 4 - Python to JavaScript Converter") as demo:

    gr.Markdown("# Week 4 — Python to JavaScript Converter")

    model_dd = gr.Dropdown(
        choices=list(MODELS.keys()),
        value=list(MODELS.keys())[0],
        label="Model"
    )

    with gr.Row():

        python_code = gr.Code(
            label="Python Code",
            language="python",
            value=EXAMPLE_CODE
        )

        js_code = gr.Code(
            label="JavaScript Code",
            language="javascript"
        )

    with gr.Row():

        run_py_btn = gr.Button("Run Python")
        convert_btn = gr.Button("Convert to JavaScript")
        run_js_btn = gr.Button("Run JavaScript")

    with gr.Row():

        python_output = gr.Textbox(label="Python Output")
        js_output = gr.Textbox(label="JavaScript Output")

    convert_btn.click(
        fn=convert_to_javascript,
        inputs=[model_dd, python_code],
        outputs=[js_code]
    )

    run_py_btn.click(
        fn=run_python,
        inputs=[python_code],
        outputs=[python_output]
    )

    run_js_btn.click(
        fn=run_javascript,
        inputs=[js_code],
        outputs=[js_output]
    )

demo.launch()